# 基于 MindSpore 的 TorchDrug 论文简化复现

我选的论文是 **TorchDrug: A Powerful and Flexible Machine Learning Platform for Drug Discovery**。这个 Notebook 主要记录我把论文里的分子性质预测流程简化后，用 MindSpore 重新实现 GIN/GAT 的过程。这里没有直接调用 PyTorch 或 TorchDrug 的训练代码，只借鉴了 TorchDrug 的图表示和特征设计。

## 1. 检查项目路径

先把项目根目录加入 `sys.path`，这样后面的代码可以直接导入 `src` 里的数据处理、模型和训练函数。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

## 2. 导入依赖和本项目代码

这一部分主要导入 MindSpore、NumPy 和我自己写的几个模块。`dataset.py` 负责数据读取和分子图构造，`models.py` 里是 GIN/GAT，`trainer.py` 里放训练和评估循环。

In [ ]:
import mindspore as ms
import numpy as np

from src.dataset import load_dataset, split_dataset
from src.models import build_model
from src.trainer import TrainConfig, fit, evaluate

ms.set_context(mode=ms.PYNATIVE_MODE, device_target="CPU")
ms.set_seed(0)
np.random.seed(0)

## 3. 读取数据并生成分子图

这里先用 BACE 做演示。代码会下载公开 CSV，然后用 RDKit 把 SMILES 转成分子图。节点特征和边特征参考了 TorchDrug 的默认设置。要跑 HIV 时，把 `dataset_name` 改成 `"hiv"` 即可。

In [ ]:
dataset_name = "bace"
dataset = load_dataset(dataset_name, PROJECT_ROOT / "data")

print("Dataset:", dataset.name)
print("Num molecules:", len(dataset))
print("Node feature dim:", dataset.node_feature_dim)
print("Edge feature dim:", dataset.edge_feature_dim)
print("First SMILES:", dataset[0]["smiles"])
print("First graph nodes:", dataset[0]["node_feature"].shape[0])

## 4. 划分训练集、验证集和测试集

BACE 这里使用 scaffold split，比例是 8:1:1。这样测试集和训练集的分子骨架差异更大，比普通随机划分更能看出模型对新结构的泛化情况。

In [ ]:
train_set, valid_set, test_set = split_dataset(dataset, split="scaffold", seed=0)
print("Train / Valid / Test:", len(train_set), len(valid_set), len(test_set))

## 5. 构建 GIN 模型

GIN 的核心是对邻居信息做求和聚合，再用 MLP 更新节点表示。这里的 `torchdrug_like` 版本加入了边特征、BatchNorm、shortcut、concat hidden 和 sum readout，让结构尽量贴近 TorchDrug 中的写法。

In [ ]:
gin_model = build_model(
    "gin",
    input_dim=dataset.node_feature_dim,
    edge_input_dim=dataset.edge_feature_dim,
    hidden_dim=128,
    num_layer=4,
    num_head=4,
    dropout=0.1,
    variant="torchdrug_like",
)
print(gin_model)

## 6. 训练并评估 GIN

为了让 Notebook 跑得快一点，这里只训练 3 个 epoch。正式写报告时我在终端里用 `run_experiment.py` 跑了更多轮数，并把结果记录到 CSV。

In [ ]:
config = TrainConfig(epoch=3, batch_size=32, lr=1e-3, seed=0)
valid_metrics, test_metrics, selected_epoch = fit(gin_model, train_set, valid_set, test_set, config)

print("Selected epoch:", selected_epoch)
print("Best valid:", valid_metrics)
print("Matched test:", test_metrics)

## 7. 构建并训练 GAT 模型

GAT 会给不同邻居分配不同注意力权重。这里同样保留边特征输入，把化学键信息加到 attention 计算里。

In [ ]:
gat_model = build_model(
    "gat",
    input_dim=dataset.node_feature_dim,
    edge_input_dim=dataset.edge_feature_dim,
    hidden_dim=128,
    num_layer=4,
    num_head=4,
    dropout=0.1,
    variant="torchdrug_like",
)

config = TrainConfig(epoch=3, batch_size=32, lr=1e-3, seed=0)
valid_metrics, test_metrics, selected_epoch = fit(gat_model, train_set, valid_set, test_set, config)

print("Selected epoch:", selected_epoch)
print("Best valid:", valid_metrics)
print("Matched test:", test_metrics)

## 8. 完整实验的运行方式

Notebook 主要用来展示流程和检查代码能不能串起来。完整实验我更建议用 `run_experiment.py` 跑，因为它会自动保存每组实验的参数和结果。

In [ ]:
# 下面几行我平时是在终端里跑的；在 Notebook 里想跑时把前面的 # 去掉即可。
import os

os.chdir(PROJECT_ROOT)
print("Current directory:", Path.cwd())

# !python run_experiment.py --dataset bace --model gin --epoch 100 --batch_size 256 --seed 0 --device_target GPU
# !python run_experiment.py --dataset bace --model gat --epoch 100 --batch_size 256 --seed 0 --device_target GPU
# !python run_experiment.py --dataset hiv --model gin --epoch 100 --batch_size 512 --seed 0 --split random --device_target GPU
# !python run_experiment.py --dataset hiv --model gat --epoch 100 --batch_size 512 --seed 0 --split random --device_target GPU


## 9. 结果记录

终端脚本会把结果追加到 `results/experiment_results.csv`。表里会记录数据集、模型、随机种子、验证集 AUROC/AUPRC 和测试集 AUROC/AUPRC，后面写报告时直接整理这个表就可以。